# RAG Indexing Pipeline — Google Colab

Embeds all chunks (BGE-M3) on Colab GPU and uploads
to a local Qdrant instance. Fully **resumable** — re-run Cell 4 any time to pick
up from where it stopped.

---

### How to use

| Goal | Cells to run |
|---|---|
| **Fresh start** (no prior data) | 1 → 2 → 3 → 4 → 5 |
| **Resume** after a crash | 2 → 4 → 5 |

**Before running anything:** set runtime to T4 GPU  
`Runtime → Change runtime type → T4 GPU`

---

### Expected final counts
| Collection | Points |
|---|---|
| `documentation_child_chunks` | integer |
| `documentation_parent_chunks` | integer |


## Cell 1 — Upload chunks.json
From your local machine.  
**Skip if the file is already present** (e.g. from a previous session that didn't disconnect).


In [6]:
import os
import shutil
import json
import glob
from google.colab import files as colab_files
from collections import Counter

# Define the target directory path
CHUNKS_DIR = "data/processed/"
os.makedirs(CHUNKS_DIR, exist_ok=True)

# 1. Custom Retriever: Check if a matching file already exists in the destination folder
existing_matches = glob.glob(os.path.join(CHUNKS_DIR, "chunks*.json"))

if existing_matches:
    CHUNKS_PATH = existing_matches[0]
    print(f"File already present at {CHUNKS_PATH} — skipping upload.")
else:
    # Check if a matching file is sitting in the root folder before triggering the picker
    local_matches = glob.glob("chunks*.json")

    if local_matches:
        source_fname = local_matches[0]
        print(f"Found existing local file matching pattern: {source_fname}")
    else:
        # 2. Custom Uploader: Trigger file picker
        print("Select your chunks*.json file in the file picker...")
        uploaded = colab_files.upload()
        if not uploaded:
            raise FileNotFoundError("No file was uploaded.")

        # Filter uploaded keys to find the one matching your pattern
        uploaded_matches = [k for k in uploaded.keys() if k.startswith("chunks") and k.endswith(".json")]
        if not uploaded_matches:
            raise ValueError("Uploaded file does not start with 'chunks' and end with '.json'")

        source_fname = uploaded_matches[0]

    # Define the final destination path, preserving the original dynamic filename
    CHUNKS_PATH = os.path.join(CHUNKS_DIR, source_fname)

    # Move the file to the target directory
    shutil.move(source_fname, CHUNKS_PATH)
    print(f"Saved to {CHUNKS_PATH}")

# 3. Access file from the final destination path
print(f"Accessing file from: {CHUNKS_PATH}")
with open(CHUNKS_PATH) as f:
    all_chunks = json.load(f)

parents  = [c for c in all_chunks if c["type"] == "parent"]
children = [c for c in all_chunks if c["type"] == "child"]
print(f"\n{os.path.basename(CHUNKS_PATH)}: {len(all_chunks)} total  —  {len(parents)} parents, {len(children)} children")

EXPECTED_CHILDREN = len(children)
EXPECTED_PARENTS = len(parents)

# Sanity check - Duplicate ids
ids = [c["id"] for c in all_chunks]
dupes = {id_: count for id_, count in Counter(ids).items() if count > 1}
print(f"Unique IDs:     {len(set(ids))}")
if len(dupes) > 0:
  raise ValueError(f"Duplicate chunks detected: {len(dupes)} duplicates found. Please verify this in chunking-pipeline.")

File already present at data/processed/chunks_d27603b_20260529.json — skipping upload.
Accessing file from: data/processed/chunks_d27603b_20260529.json

chunks_d27603b_20260529.json: 25556 total  —  7825 parents, 17731 children
Unique IDs:     25556


## Cell 2 — Configuration & Dependencies
**Run once per session.** All tunable constants live here.



In [2]:
# Install FlagEmbedding for BGE-M3 instead of fastembed
%pip install -q qdrant-client FlagEmbedding

# ── Constants ─────────────────────────────────────────────────────────────────
QDRANT_VERSION       = "v1.13.2"            # keep in sync with docker-compose.yml
QDRANT_URL           = "http://localhost:6333"
QDRANT_STORAGE       = "/content/qdrant_storage"
QDRANT_LOG           = "/content/qdrant.log"

# These must match init_vectordB.py and index_chunks.py exactly
CHILDREN_COLLECTION  = "documentation_child_chunks"
PARENTS_COLLECTION   = "documentation_parent_chunks"

# BGE-M3 acts as a single model managing both Dense and Sparse representations
BGE_M3_MODEL         = "BAAI/bge-m3"
DENSE_DIM            = 1024                 # BGE-M3 outputs 1024-dimensional dense vectors

EMBED_BATCH_SIZE     = 64    # texts per embedding forward pass
UPLOAD_BATCH_SIZE    = 256   # points per Qdrant upload call


print("✅ Configuration and BGE-M3 requirements loaded.")

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.3 MB/s eta 0:00:00
✅ Configuration and BGE-M3 requirements loaded.


## Cell 3 — Start Qdrant
**Safe to re-run.** Kills any existing Qdrant process and starts a fresh one.

Always starts with **empty storage** — resume logic lives in Cell 4 (Qdrant-side,
not filesystem-side). This avoids the cgroup panic that crashed the previous attempt
when loading pre-existing segment files.

> **Why not restore from zip?**  
> The previous session showed that Qdrant v1.13.2 panics on Colab when it loads
> *existing* storage (segments trigger an mmap path that reads a cgroup memory file
> that doesn't exist in Colab's VM, and `/sys/fs/cgroup` is read-only so it can't
> be faked). Empty storage starts fine because no segments exist yet. The resume
> logic in Cell 4 talks to a *running* Qdrant over HTTP and re-uploads only missing
> points — no filesystem restore needed.


In [3]:
import os, subprocess, time, urllib.request

def _stop_qdrant():
    """Terminate any running Qdrant process so we can restart cleanly."""
    proc = globals().get("qdrant_process")
    if proc and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
        print("Stopped existing Qdrant process.")
    os.system("pkill -f './qdrant' 2>/dev/null")
    time.sleep(1)


def _download_qdrant_binary():
    if os.path.exists("./qdrant"):
        print("Qdrant binary already present — skipping download.")
        return
    url = (f"https://github.com/qdrant/qdrant/releases/download/"
           f"{QDRANT_VERSION}/qdrant-x86_64-unknown-linux-musl.tar.gz")
    print(f"Downloading Qdrant {QDRANT_VERSION}...")
    urllib.request.urlretrieve(url, "qdrant.tar.gz")
    os.system("tar -xzf qdrant.tar.gz && chmod +x qdrant")
    print("Binary ready.")


def _start_qdrant():
    """
    Start Qdrant with empty storage.

    Key env vars:
      QDRANT__STORAGE__MMAP_THRESHOLD_KB = 99999999
        Forces Qdrant to keep everything in RAM rather than mmap-ing to disk.
        mmap is what triggers the cgroup memory.high read that panics on Colab.
        With a threshold higher than any segment will ever be, mmap is never used.

      QDRANT__STORAGE__ON_DISK_PAYLOAD = false
        Keeps payloads in RAM too, consistent with the above.
    """
    os.makedirs(QDRANT_STORAGE, exist_ok=True)

    env = os.environ.copy()
    env["QDRANT__STORAGE__STORAGE_PATH"]       = QDRANT_STORAGE
    # Keep all data in RAM — prevents mmap, which is what reads the missing
    # cgroup memory.high file and causes the Colab panic.
    # SCALING NOTE: safe for this dataset (~27K chunks, ~150-200 MB RAM).
    # Do NOT use on datasets with millions of chunks — Qdrant will OOM.
    # If you outgrow Colab RAM, the correct fix is to run Qdrant in Docker
    # on a VM (GCE/EC2) where cgroups are properly configured.
    env["QDRANT__STORAGE__MMAP_THRESHOLD_KB"]  = "99999999"   # never mmap → no cgroup read
    env["QDRANT__STORAGE__ON_DISK_PAYLOAD"]    = "false"
    env["QDRANT__SERVICE__MAX_REQUEST_SIZE_MB"] = "32"

    log = open(QDRANT_LOG, "w")
    proc = subprocess.Popen(["./qdrant"], env=env, stdout=log, stderr=log)
    globals()["qdrant_process"] = proc

    print("Waiting for Qdrant", end="")
    for _ in range(60):
        time.sleep(1)
        try:
            urllib.request.urlopen(f"{QDRANT_URL}/", timeout=2)
            print(" ready!")
            return
        except Exception:
            print(".", end="", flush=True)

    # Startup failed — print log for diagnosis
    proc.terminate()
    log.flush()
    print("\n=== Qdrant log ===")
    with open(QDRANT_LOG) as f:
        print(f.read())
    raise RuntimeError("Qdrant failed to start within 60s. See log above.")


# ── Run ───────────────────────────────────────────────────────────────────────
_stop_qdrant()
_download_qdrant_binary()
_start_qdrant()
print(f"\nQdrant running at {QDRANT_URL}")


Binary ready.
Waiting for Qdrant ready!

Qdrant running at http://localhost:6333


## Cell 4 — Initialize Collections
**Idempotent** — skips creation if a collection already exists.  
On a fresh Qdrant start both collections will be created. On resume they already
exist and are just reported with their current point counts.


In [4]:
from qdrant_client import QdrantClient, models

client = QdrantClient(url=QDRANT_URL)


def _create_if_missing(name, vectors_config, sparse_vectors_config=None, payload_indexes=None):
    if client.collection_exists(name):
        info = client.get_collection(name)
        print(f"  '{name}' already exists ({info.points_count} points) — skipping creation.")
        return
    client.create_collection(
        collection_name=name,
        vectors_config=vectors_config,
        sparse_vectors_config=sparse_vectors_config,
    )
    for field in (payload_indexes or []):
        client.create_payload_index(name, field, models.PayloadSchemaType.KEYWORD)
    print(f"  ✅ '{name}' created.")


_create_if_missing(
    CHILDREN_COLLECTION,
    vectors_config={"dense": models.VectorParams(size=DENSE_DIM, distance=models.Distance.COSINE)},
    sparse_vectors_config={"sparse": models.SparseVectorParams()},
    payload_indexes=["framework", "content_category", "parent_id"],
)

_create_if_missing(
    PARENTS_COLLECTION,
    vectors_config={"dense": models.VectorParams(size=DENSE_DIM, distance=models.Distance.COSINE)},
    payload_indexes=["framework", "doc_id"],
)

print("\nCurrent state:")
for name in [CHILDREN_COLLECTION, PARENTS_COLLECTION]:
    info = client.get_collection(name)
    print(f"  {name}: {info.points_count} points")


/usr/local/lib/python3.12/dist-packages/qdrant_client/qdrant_remote.py:282: UserWarning: Qdrant client version 1.18.0 is incompatible with server version 1.13.2. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


  ✅ 'documentation_child_chunks' created.
  ✅ 'documentation_parent_chunks' created.

Current state:
  documentation_child_chunks: 0 points
  documentation_parent_chunks: 0 points


## Cell 5 — Index Chunks (resumable)

**This is the only cell you ever need to re-run after a crash.**

How resume works:
1. Scrolls both Qdrant collections to collect all existing point UUIDs (~1 s)
2. Filters `chunks.json` to only the chunks whose UUID is not yet in Qdrant
3. Embeds and uploads only those missing chunks

Because point IDs are deterministic (UUID5 of the chunk's string ID), the
same chunk always maps to the same UUID. Re-running this cell is therefore
safe and idempotent — already-indexed points are never touched.

**Expected time from scratch on T4 GPU: ~15–20 min**


In [8]:
import json, os, uuid, time
import torch  # Used to verify GPU availability
from qdrant_client import QdrantClient, models
from FlagEmbedding import BGEM3FlagModel

client = QdrantClient(url=QDRANT_URL)


# ── Helpers ───────────────────────────────────────────────────────────────────

def _deterministic_uuid(s: str) -> str:
    """Stable UUID from chunk string ID — same input always yields same UUID."""
    return str(uuid.uuid5(uuid.NAMESPACE_DNS, s))


def _batch(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i : i + n]


def _get_existing_ids(collection: str) -> set:
    """Scroll entire collection and return set of all point UUIDs."""
    existing, offset = set(), None
    while True:
        results, offset = client.scroll(
            collection_name=collection,
            offset=offset,
            limit=1000,
            with_payload=False,
            with_vectors=False,
        )
        for r in results:
            existing.add(r.id)
        if offset is None:
            break
    return existing


# ── GPU check ────────────────────────────────────────────────────────────────
is_cuda = torch.cuda.is_available()
if not is_cuda:
    print("⚠️ CUDA not available — running on CPU (will be significantly slower)")
else:
    print("✅ CUDA/GPU environment detected — Accelerating indexing with FP16.")


# ── Load chunks ───────────────────────────────────────────────────────────────
with open(CHUNKS_PATH) as f:
    all_chunks = json.load(f)

parent_chunks = [c for c in all_chunks if c["type"] == "parent"]
child_chunks  = [c for c in all_chunks if c["type"] == "child"]
print(f"\nTotal: {len(child_chunks)} children, {len(parent_chunks)} parents")


# ── Find what is missing ──────────────────────────────────────────────────────
print("\nChecking existing index state (scrolling collections)...")
existing_children = _get_existing_ids(CHILDREN_COLLECTION)
existing_parents  = _get_existing_ids(PARENTS_COLLECTION)
print(f"Already indexed — children: {len(existing_children)}, parents: {len(existing_parents)}")

missing_children = [c for c in child_chunks  if _deterministic_uuid(c["id"]) not in existing_children]
missing_parents  = [c for c in parent_chunks if _deterministic_uuid(c["id"]) not in existing_parents]
print(f"Still to index  — children: {len(missing_children)}, parents: {len(missing_parents)}")

if not missing_children and not missing_parents:
    print("\n✅ Nothing to do — all chunks already indexed!")
else:
    # ── Load BGE-M3 model ─────────────────────────────────────────────────────
    print("\nLoading unified BGE-M3 embedding model...")
    # use_fp16=True targets the GPU context natively and optimizes memory limits
    bge_model = BGEM3FlagModel(BGE_M3_MODEL, use_fp16=is_cuda)
    print("Model loaded into memory.\n")

    # ── Index missing children ────────────────────────────────────────────────
    if missing_children:
        total      = len(missing_children)
        t_start    = time.time()
        print(f"Indexing {total} children...")

        for batch_idx, chunk_batch in enumerate(_batch(missing_children, EMBED_BATCH_SIZE)):
            texts = [c["text"] for c in chunk_batch]

            # Execute batch-wise encoding for both dense and sparse contexts
            embeddings = bge_model.encode(texts, return_dense=True, return_sparse=True)
            dense_vectors  = embeddings['dense_vecs']
            sparse_vectors = embeddings['lexical_weights'] # List of dicts: [{'token': weight}, ...]

            points = []
            for c, dv, sv in zip(chunk_batch, dense_vectors, sparse_vectors):

                # Convert string tokens to IDs and aggregate weights for any duplicate integer IDs
                """
                By routing the token-to-ID conversion through an intermediary dictionary (idx_to_weight),
                Python natively deduplicates the integer IDs. If two subwords collapse into the same ID,
                their semantic importance weights are summed up together perfectly. This guarantees a clean,
                100% unique list of indices that Qdrant will accept without errors.
                """
                idx_to_weight = {}
                for token, weight in sv.items():
                    token_id = bge_model.tokenizer.convert_tokens_to_ids(token)
                    # Sum the weights if two string tokens map to the same vocabulary ID
                    idx_to_weight[token_id] = idx_to_weight.get(token_id, 0.0) + float(weight)

                indices = list(idx_to_weight.keys())
                values = list(idx_to_weight.values())

                points.append(
                    models.PointStruct(
                        id=_deterministic_uuid(c["id"]),
                        vector={
                            "dense":  dv.tolist(),
                            "sparse": models.SparseVector(
                                indices=indices,
                                values=values,
                            ),
                        },
                        payload={
                            "chunk_id":         c["id"],
                            "text":             c["text"],
                            "parent_id":        c["metadata"].get("parent_id"),
                            "framework":        c["metadata"].get("framework"),
                            "content_category": c["metadata"].get("content_category"),
                            "breadcrumbs":      c["metadata"].get("breadcrumbs"),
                            "source_file":      c["metadata"].get("source_file"),
                        },
                    )
                )

            client.upload_points(collection_name=CHILDREN_COLLECTION, points=points, wait=True)

            done    = min((batch_idx + 1) * EMBED_BATCH_SIZE, total)
            elapsed = time.time() - t_start
            rate    = done / elapsed if elapsed > 0 else 0
            eta     = (total - done) / rate if rate > 0 else 0
            print(f"  Children: {done:>6}/{total}  |  {rate:.0f} chunks/s  |  ETA {eta/60:.1f} min")

        print(f"✅ Children done in {(time.time()-t_start)/60:.1f} min")

    # ── Index missing parents ─────────────────────────────────────────────────
    if missing_parents:
        total   = len(missing_parents)
        t_start = time.time()
        print(f"\nIndexing {total} parents...")

        for batch_idx, chunk_batch in enumerate(_batch(missing_parents, EMBED_BATCH_SIZE)):
            texts         = [c["text"] for c in chunk_batch]

            # For parents, compute only dense embeddings to minimize compute footprint
            embeddings    = bge_model.encode(texts, return_dense=True, return_sparse=False)
            dense_vectors = embeddings['dense_vecs']

            points = [
                models.PointStruct(
                    id=_deterministic_uuid(c["id"]),
                    vector={"dense": dv.tolist()},
                    payload={
                        "doc_id":          c["id"],
                        "text":            c["text"],
                        "framework":       c["metadata"].get("framework"),
                        "global_title":    c["metadata"].get("global_title"),
                        "section_heading": c["metadata"].get("section_heading"),
                        "source_file":     c["metadata"].get("source_file"),
                    },
                )
                for c, dv in zip(chunk_batch, dense_vectors)
            ]

            client.upload_points(collection_name=PARENTS_COLLECTION, points=points, wait=True)

            done    = min((batch_idx + 1) * EMBED_BATCH_SIZE, total)
            elapsed = time.time() - t_start
            rate    = done / elapsed if elapsed > 0 else 0
            eta     = (total - done) / rate if rate > 0 else 0
            print(f"  Parents: {done:>5}/{total}  |  {rate:.0f} chunks/s  |  ETA {eta/60:.1f} min")

        print(f"✅ Parents done in {(time.time()-t_start)/60:.1f} min")

    print("\n🎉 Indexing complete. Run Cell 6 to verify and download.")

✅ CUDA/GPU environment detected — Accelerating indexing with FP16.

Total: 17731 children, 7825 parents

Checking existing index state (scrolling collections)...
Already indexed — children: 0, parents: 0
Still to index  — children: 17731, parents: 7825

Loading unified BGE-M3 embedding model...


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model loaded into memory.

Indexing 17731 children...
  Children:     64/17731  |  24 chunks/s  |  ETA 12.3 min
  Children:    128/17731  |  22 chunks/s  |  ETA 13.2 min
  Children:    192/17731  |  25 chunks/s  |  ETA 11.7 min
  Children:    256/17731  |  28 chunks/s  |  ETA 10.5 min
  Children:    320/17731  |  30 chunks/s  |  ETA 9.8 min
  Children:    384/17731  |  31 chunks/s  |  ETA 9.4 min
  Children:    448/17731  |  30 chunks/s  |  ETA 9.5 min
  Children:    512/17731  |  28 chunks/s  |  ETA 10.2 min
  Children:    576/17731  |  27 chunks/s  |  ETA 10.6 min
  Children:    640/17731  |  26 chunks/s  |  ETA 10.9 min
  Children:    704/17731  |  25 chunks/s  |  ETA 11.1 min
  Children:    768/17731  |  25 chunks/s  |  ETA 11.3 min
  Children:    832/17731  |  25 chunks/s  |  ETA 11.4 min
  Children:    896/17731  |  24 chunks/s  |  ETA 11.6 min
  Children:    960/17731  |  24 chunks/s  |  ETA 11.7 min
  Children:   1024/17731  |  23 chunks/s  |  ETA 12.0 min
  Children:   1088/17

## Cell 6 — Verify, Flush & Download

Checks point counts, **forces Qdrant to flush WAL to disk segments** (the step
that was missing in the previous session — without this, vectors live only in RAM
and the zip contains only WAL scaffolding with no actual data), then shuts down
cleanly, zips, and downloads.

**After download, on your local machine:**
```bash
cd docker && docker compose down          # stop local Qdrant
rm -rf data/qdrant_dB/*                   # clear existing storage
unzip qdrant_storage.zip -d data/qdrant_dB/
docker compose up -d
```
Verify at http://localhost:6333/dashboard — both collections should show correct
counts immediately with no re-indexing.


In [9]:
import os, shutil, time
from qdrant_client import QdrantClient, models
from google.colab import files as colab_files

client = QdrantClient(url=QDRANT_URL)

# ── Step 1: Verify counts ─────────────────────────────────────────────────────
print("Verifying point counts...")
# children_count = client.get_collection(CHILDREN_COLLECTION).points_count
# parents_count  = client.get_collection(PARENTS_COLLECTION).points_count
children_count = client.count(CHILDREN_COLLECTION, exact=True).count
parents_count  = client.count(PARENTS_COLLECTION,  exact=True).count
print(f"  {CHILDREN_COLLECTION}: {children_count} / {EXPECTED_CHILDREN} expected")
print(f"  {PARENTS_COLLECTION}:  {parents_count}  / {EXPECTED_PARENTS} expected")

if children_count < EXPECTED_CHILDREN or parents_count < EXPECTED_PARENTS:
    missing_c = EXPECTED_CHILDREN - children_count
    missing_p = EXPECTED_PARENTS  - parents_count
    print(f"\n⚠️  Incomplete — {missing_c} children and {missing_p} parents still missing.")
    print("Re-run Cell 5 to index remaining chunks, then re-run this cell.")
    raise SystemExit("Aborting download — indexing is not complete.")

print("\n✅ All points indexed correctly!")

# ── Step 2: Force WAL → disk flush ───────────────────────────────────────────
# Without this step Qdrant keeps vectors in RAM. The zip would contain only
# WAL scaffolding (~200 MB of near-empty files) and be useless for restore.
# Setting indexing_threshold=0 forces the optimizer to flush all segments NOW.
print("\nForcing WAL flush to disk (setting indexing_threshold=0)...")
for name in [CHILDREN_COLLECTION, PARENTS_COLLECTION]:
    client.update_collection(
        collection_name=name,
        optimizer_config=models.OptimizersConfigDiff(indexing_threshold=0),
    )

print("Waiting for optimizer to finish flushing", end="")
for _ in range(120):   # up to 2 minutes
    time.sleep(2)
    statuses = []
    for name in [CHILDREN_COLLECTION, PARENTS_COLLECTION]:
        info = client.get_collection(name)
        statuses.append(info.status)
    if all(s.value == "green" for s in statuses):
        print(" done!")
        break
    print(".", end="", flush=True)
else:
    print("\n⚠️  Optimizer did not finish within 2 min. Proceeding anyway — zip may be partial.")

# ── Step 3: Confirm segment files exist on disk ───────────────────────────────
print("\nChecking segment files on disk (files > 1 MB):")
total_size = 0
for root, dirs, files in os.walk(QDRANT_STORAGE):
    for fname in files:
        fpath = os.path.join(root, fname)
        size  = os.path.getsize(fpath)
        total_size += size
        if size > 1_000_000:
            rel = fpath.replace(QDRANT_STORAGE, "")
            print(f"  {size/1024/1024:.1f} MB — {rel}")
print(f"Total storage size: {total_size/1024/1024:.1f} MB")

if total_size < 500_000_000:   # expect > 500 MB for a proper index
    print("⚠️  Storage seems small — vectors may not have flushed fully.")
else:
    print("✅ Storage size looks correct.")

# ── Step 4: Reset optimizer threshold to default ────────────────────────────
# indexing_threshold=0 is saved into collection metadata inside the storage
# snapshot. If left at 0, any future upsert after local restore will
# immediately trigger the background optimizer, degrading write performance.
# Reset to Qdrant's default (20000) so the restored collections behave normally.
print("\nResetting optimizer threshold to default (20000)...")
for name in [CHILDREN_COLLECTION, PARENTS_COLLECTION]:
    client.update_collection(
        collection_name=name,
        optimizer_config=models.OptimizersConfigDiff(indexing_threshold=20000),
    )
print("✅ Optimizer config reset.")

# ── Step 5: Shut down Qdrant cleanly ──────────────────────────────────────────
print("\nShutting down Qdrant...")
proc = globals().get("qdrant_process")
if proc and proc.poll() is None:
    proc.terminate()
    try:
        proc.wait(timeout=15)
    except Exception:
        proc.kill()
print("Qdrant stopped.")

# ── Step 6: Zip and download ──────────────────────────────────────────────────
zip_path = "/content/qdrant_storage"
print("Zipping storage...")
shutil.make_archive(zip_path, "zip", QDRANT_STORAGE)
size_mb = os.path.getsize(f"{zip_path}.zip") / (1024 * 1024)
print(f"Archive ready: qdrant_storage.zip ({size_mb:.1f} MB)")

colab_files.download(f"{zip_path}.zip")


Verifying point counts...
  documentation_child_chunks: 17731 / 17731 expected
  documentation_parent_chunks:  7825  / 7825 expected

✅ All points indexed correctly!

Forcing WAL flush to disk (setting indexing_threshold=0)...
Waiting for optimizer to finish flushing

/usr/local/lib/python3.12/dist-packages/qdrant_client/qdrant_remote.py:282: UserWarning: Qdrant client version 1.18.0 is incompatible with server version 1.13.2. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


 done!

Checking segment files on disk (files > 1 MB):
  32.0 MB — /collections/documentation_parent_chunks/0/wal/open-2
  32.0 MB — /collections/documentation_parent_chunks/0/wal/closed-0
  32.0 MB — /collections/documentation_parent_chunks/0/wal/open-3
  1.0 MB — /collections/documentation_parent_chunks/0/segments/60e2520c-784b-4742-855a-c34b4d004626/vector_storage-dense/deleted/flags_a.dat
  32.0 MB — /collections/documentation_parent_chunks/0/segments/60e2520c-784b-4742-855a-c34b4d004626/vector_storage-dense/vectors/chunk_0.mmap
  1.0 MB — /collections/documentation_parent_chunks/0/segments/6fb6f004-1ed0-49f0-86a4-58062b061552/vector_storage-dense/deleted/flags_a.dat
  32.0 MB — /collections/documentation_parent_chunks/0/segments/6fb6f004-1ed0-49f0-86a4-58062b061552/vector_storage-dense/vectors/chunk_0.mmap
  32.0 MB — /collections/documentation_child_chunks/0/wal/open-4
  32.0 MB — /collections/documentation_child_chunks/0/wal/closed-127
  32.0 MB — /collections/documentation_chil

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>